## Experiment to identify memory capacity in the autoencoder RNN hidden state

### Recipe: experiment 1
- generate completely random sequence
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

### Recipe: experiment 2
- generate a sequence with pattern (1,2,3,4,5, 1,2,3,4,5, 1,2,3,4,5....)
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

In [39]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [40]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [41]:
def get_patterned_sequence(total_samples, token_number=7):
    sequence = []
    for i in range(total_samples):
        sequence.append(chr((i % token_number) + ord('A')))
    return sequence

In [42]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [43]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-64
      
            self.y[ii] = \
                ord(data[ii+jj+1])-64

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [44]:
# tokens start from 1
class Dataset_reconstructer(Dataset):
    def __init__(self, hidden_states, data, past_recall=1, short_term_memory=1):
        
        self.X = np.array(hidden_states)
        self.y = np.vectorize(ord)(data) - 64
        self.short_term_memory = short_term_memory
        self.past_recall = past_recall

        if short_term_memory == 1:
            self.y = np.concatenate(
                    (np.zeros(past_recall, dtype=int), self.y[:-past_recall])
                )

        self.X = tnsr(self.X)
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index+self.short_term_memory-self.past_recall-1]

    def __len__(self):
        return self.X.shape[0]

In [45]:
def path_finder_loss(y_pred, decoder_output, y_target, decoder_target):
    ce1 = F.cross_entropy(y_pred.view(-1, y_pred.size(-1)), y_target.view(-1))
    ce2 = F.cross_entropy(decoder_output.view(-1, decoder_output.size(-1)), decoder_target.view(-1))

    return (ce1+ce2)/2

In [46]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        self.linear = nn.Linear(hidden_size, vocab_size)

        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  
    
class RNNDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='relu')
        self.fc = nn.Linear(hidden_size, vocab_size)

        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded, h)
        return self.fc(out) 
    
class RNNAutoencoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.encoder = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        self.decoder = RNNDecoder(vocab_size, embedding_dim, hidden_size)
    
    def forward(self, x, decoder_input, h=None):
        next_token, h = self.encoder(x, h)
        decoder_output = self.decoder(decoder_input, h)
        return next_token, decoder_output, h

In [47]:
class classifier(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        out = nn.functional.relu(self.linear1(x))
        out = self.linear2(x)

        return out  

In [50]:

reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNAutoencoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_random_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        decoder_correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            decoder_input = torch.cat([torch.zeros((1, 1), dtype=torch.long), X[:, :-1]], dim=1)
            decoder_target = X.flip(1)

            optimizer.zero_grad()

            if total == 0:
                y_pred, decoder_output, h = model(X, decoder_input)
            else:
                y_pred, decoder_output, h = model(X, decoder_input, hidden)

            loss = criterion(y_pred[0], decoder_output[0], y[0], decoder_target[0])     
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                if torch.sum(decoder_output.argmax(2) == decoder_target)==short_term_memory:
                        decoder_correct[total%1000] = 1
                else:
                    decoder_correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                test_acc_decoder.append(
                    np.sum(decoder_correct)/total if total<1000 else np.sum(decoder_correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}, decoder accuracy: {test_acc_decoder[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model.encoder(X)
                else:
                    y_pred, h = model.encoder(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_random_autoencoder.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, loss: 1.0578, accuracy: 0.1460, decoder accuracy: 0.6390
Iter : 2001, loss: 0.8766, accuracy: 0.1490, decoder accuracy: 0.9860
Iter : 3001, loss: 1.1042, accuracy: 0.1390, decoder accuracy: 1.0000
Iter : 4001, loss: 1.1448, accuracy: 0.1420, decoder accuracy: 1.0000
Iter : 5001, loss: 0.9259, accuracy: 0.1260, decoder accuracy: 1.0000
Iter : 6001, loss: 1.1563, accuracy: 0.1530, decoder accuracy: 1.0000
Iter : 7001, loss: 0.9216, accuracy: 0.1250, decoder accuracy: 0.9970
Iter : 8001, loss: 1.0350, accuracy: 0.1420, decoder accuracy: 1.0000
Iter : 9001, loss: 0.7754, accuracy: 0.1300, decoder accuracy: 1.0000
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.16it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.18it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9960988296488946
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 0.9971969166082691
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.20it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9785764340774852
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.19it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9235158674541997


In [48]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNAutoencoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_patterned_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        decoder_correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            decoder_input = torch.cat([torch.zeros((1, 1), dtype=torch.long), X[:, :-1]], dim=1)
            decoder_target = X.flip(1)

            optimizer.zero_grad()

            if total == 0:
                y_pred, decoder_output, h = model(X, decoder_input)
            else:
                y_pred, decoder_output, h = model(X, decoder_input, hidden)

            loss = criterion(y_pred[0], decoder_output[0], y[0], decoder_target[0])     
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                if torch.sum(decoder_output.argmax(2) == decoder_target)==short_term_memory:
                        decoder_correct[total%1000] = 1
                else:
                    decoder_correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                test_acc_decoder.append(
                    np.sum(decoder_correct)/total if total<1000 else np.sum(decoder_correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}, decoder accuracy: {test_acc_decoder[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model.encoder(X)
                else:
                    y_pred, h = model.encoder(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_patterned_autoencoder.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, loss: 0.0060, accuracy: 0.9470, decoder accuracy: 0.9450
Iter : 2001, loss: 0.0016, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 3001, loss: 0.0007, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 4001, loss: 0.0004, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 5001, loss: 0.0002, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 6001, loss: 0.0001, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 7001, loss: 0.0001, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 8001, loss: 0.0000, accuracy: 1.0000, decoder accuracy: 1.0000
Iter : 9001, loss: 0.0000, accuracy: 1.0000, decoder accuracy: 1.0000
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.16it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9994998499549865
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.15it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9994998499549865
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.20it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.18it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0


In [49]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 6e-4
repitition = []
acc = []
test_acc = []
test_acc_decoder = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNAutoencoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = path_finder_loss

        data = get_sequence(total_samples, 2, 3, train_percent=1.0)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        decoder_correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            decoder_input = torch.cat([torch.zeros((1, 1), dtype=torch.long), X[:, :-1]], dim=1)
            decoder_target = X.flip(1)

            optimizer.zero_grad()

            if total == 0:
                y_pred, decoder_output, h = model(X, decoder_input)
            else:
                y_pred, decoder_output, h = model(X, decoder_input, hidden)

            loss = criterion(y_pred[0], decoder_output[0], y[0], decoder_target[0])     
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0

                if torch.sum(decoder_output.argmax(2) == decoder_target)==short_term_memory:
                        decoder_correct[total%1000] = 1
                else:
                    decoder_correct[total%1000] = 0

                test_acc.append(
                    np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                )
                test_acc_decoder.append(
                    np.sum(decoder_correct)/total if total<1000 else np.sum(decoder_correct)/1000
                )
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}, decoder accuracy: {test_acc_decoder[-1]:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model.encoder(X)
                else:
                    y_pred, h = model.encoder(X, h)

                hidden_states.append(
                    h[0][0]
                )

        data_ = []
        for ch in data:
            data_.append(ch)

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data_, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_hard_patterned_autoencoder.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, loss: 0.9043, accuracy: 0.4330, decoder accuracy: 0.7530
Iter : 2001, loss: 0.4697, accuracy: 0.6080, decoder accuracy: 1.0000
Iter : 3001, loss: 1.1319, accuracy: 0.6620, decoder accuracy: 1.0000
Iter : 4001, loss: 0.1983, accuracy: 0.6740, decoder accuracy: 1.0000
Iter : 5001, loss: 0.0697, accuracy: 0.6420, decoder accuracy: 1.0000
Iter : 6001, loss: 1.5542, accuracy: 0.6750, decoder accuracy: 1.0000
Iter : 7001, loss: 0.5364, accuracy: 0.6590, decoder accuracy: 0.9980
Iter : 8001, loss: 0.2683, accuracy: 0.6720, decoder accuracy: 1.0000
Iter : 9001, loss: 0.6600, accuracy: 0.6570, decoder accuracy: 0.9980
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.17it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.18it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9722916875062518
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.12it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9765742316548203
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:08<00:00,  1.11it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9197116828511362
